# B3_research_benchmarks: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/bonus/B3_research_benchmarks/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Compute an independent distance boundary and distinguish approximation from relevance.
Concepts: ground truth and evaluation.
Run: uv run python modules/bonus/B3_research_benchmarks/examples/01_from_scratch.py
Expected: assertions pass, then measured values or a clearly labeled illustrative result.
CPU: seconds to a minute; frozen MiniLM or explicit public subset, no model downloads.
Native profiles and optional packages are described in README.md.
"""

import numpy as np


def main() -> None:
    base = np.array([[0, 0], [1, 0], [0, 1], [2, 2]], dtype=float)
    query = np.array([0.9, 0.1])
    distances = ((base - query) ** 2).sum(axis=1)
    truth = np.argsort(distances, kind="stable")[:2]
    candidate = np.array([1, 3])
    recall = np.mean(distances[candidate] <= distances[truth[-1]])
    assert recall == 0.5
    print({"truth": truth.tolist(), "candidate": candidate.tolist(), "recall": recall})
    print("No category labels are needed for ANN recall; relevance needs independent judgments.")


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## 02_production

Source: `modules/bonus/B3_research_benchmarks/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("faiss") is None or "public-benchmark" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra libs; start public-benchmark and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Evaluate native HNSW on a checksummed, licensed Fashion-MNIST subset.
    Concepts: ground truth and evaluation.
    Run: uv run python modules/bonus/B3_research_benchmarks/examples/02_production.py
    Expected: assertions pass, then measured values or a clearly labeled illustrative result.
    CPU: seconds to a minute; frozen MiniLM or explicit public subset, no model downloads.
    Native profiles and optional packages are described in README.md.
    """
    # course-extra: libs
    # course-profile: public-benchmark

    from common.research import public_benchmark, record

    def main() -> None:
        result = public_benchmark()
        print({"efSearch": result["chosen_efSearch"], "heldout": result["heldout"]})
        print(
            record(
                "public-benchmark",
                result,
                "python modules/bonus/B3_research_benchmarks/examples/02_production.py",
            )
        )

    if __name__ == "__main__":
        from common.errors import run_cli

        run_cli(main)

## 03_measure

Source: `modules/bonus/B3_research_benchmarks/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Show uncertainty and why repeated timings are not independent queries.
Concepts: ground truth and evaluation.
Run: uv run python modules/bonus/B3_research_benchmarks/examples/03_measure.py
Expected: assertions pass, then measured values or a clearly labeled illustrative result.
CPU: seconds to a minute; frozen MiniLM or explicit public subset, no model downloads.
Native profiles and optional packages are described in README.md.
"""

import numpy as np


def main() -> None:
    per_query = np.array([1.0, 0.9, 0.8, 1.0, 0.7, 1.0, 0.9, 0.8, 1.0, 0.9])
    rng = np.random.default_rng(42)
    bootstrap = [rng.choice(per_query, len(per_query), replace=True).mean() for _ in range(2000)]
    lo, hi = np.percentile(bootstrap, [2.5, 97.5])
    print(
        {
            "illustrative_recall": per_query.mean(),
            "query_bootstrap_interval": [lo, hi],
            "distinct_queries": len(per_query),
        }
    )
    assert lo <= per_query.mean() <= hi
    print(
        "These illustrative scores are not public-dataset measurements. Bootstrap queries, not repeated timestamps."
    )


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
import numpy as np

scores = np.array([1.0, 0.8, 0.6])
print(scores.mean(), len(scores))
# Repeating these queries does not create new independent judgments.

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.